In [5]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

### Q1 Spark Version

In [ ]:
print(spark.version)

In [7]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [10]:
df = df.repartition(4)
df.write.mode("overwrite").parquet('yellow/2025/11/', )

### Q3 Count records

In [13]:
df.withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .filter("pickup_date = '2025-11-15'") \
    .count()

162604

### Q4 Longest trip

In [23]:
df.agg(F.max(df.tpep_dropoff_datetime.cast('long') - df.tpep_pickup_datetime.cast('long')).alias('max_trip_duration')).show()

{"ts": "2026-02-26 22:46:58.157", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION] Cannot resolve \"CAST(tpep_dropoff_datetime AS BIGINT)\" due to data type mismatch: cannot cast \"TIMESTAMP_NTZ\" to \"BIGINT\". SQLSTATE: 42K09", "context": {"file": "line 1 in cell [24]", "line": "", "fragment": "cast", "errorClass": "DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o121.agg.\n: org.apache.spark.sql.AnalysisException: [DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION] Cannot resolve \"CAST(tpep_dropoff_datetime AS BIGINT)\" due to data type mismatch: cannot cast \"TIMESTAMP_NTZ\" to \"BIGINT\". SQLSTATE: 42K09;\n'Aggregate ['max('`-`(cast(tpep_dropoff_datetime#22 as bigint), cast(tpep_pickup_datetime#21 as bigint))) AS max_trip_duration#82]\n+- Repartition 4, true\n   +- Repartition 4, true\n      +- Relation [VendorID#20,tpep_pickup_datetime#21,tpep_dropof

AnalysisException: [DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION] Cannot resolve "CAST(tpep_dropoff_datetime AS BIGINT)" due to data type mismatch: cannot cast "TIMESTAMP_NTZ" to "BIGINT". SQLSTATE: 42K09;
'Aggregate ['max('`-`(cast(tpep_dropoff_datetime#22 as bigint), cast(tpep_pickup_datetime#21 as bigint))) AS max_trip_duration#82]
+- Repartition 4, true
   +- Repartition 4, true
      +- Relation [VendorID#20,tpep_pickup_datetime#21,tpep_dropoff_datetime#22,passenger_count#23L,trip_distance#24,RatecodeID#25L,store_and_fwd_flag#26,PULocationID#27,DOLocationID#28,payment_type#29L,fare_amount#30,extra#31,mta_tax#32,tip_amount#33,tolls_amount#34,improvement_surcharge#35,total_amount#36,congestion_surcharge#37,Airport_fee#38,cbd_congestion_fee#39] parquet
